# MedVision Smart — Notebook 04 v2 : évaluation finale sur les deux datasets

**Équipe :** Abderrahman Ghazouani · Hibatallah Hamrani · Mohammed Anas Latifi

---

Ce notebook évalue **une seule fois sur le test** les 7 algorithmes, sur **BUSI** et **Kvasir-SEG** :

| Algorithme | Entraînements |
|---|---|
| Otsu (code harmonisé de Hiba, `otsu_baseline.py`) | réglé sur la validation de chaque dataset, sans graine |
| CNN simple, U-Net, U-Net++, Attention U-Net, DeepLabV3+, SegFormer | **5 graines** chacun, produits par `run_all.py` |

Toutes les décisions (hyperparamètres, learning rate, réglage d'Otsu) ont été prises **sur la validation**. Ce notebook ne modifie aucun modèle après avoir vu le test.

### Correspondance avec les remarques du professeur

| Partie | Remarques |
|---|---|
| 1. Entraînements : hyperparamètres, complexité, coûts | 4, 5, 14 |
| 2. Courbes d'apprentissage (pertes, Dice, IoU) | 6 |
| 3. Otsu harmonisé | 9 |
| 4. Contrôle de cohérence | 2 |
| 5. Évaluation sur le test, mAP vérifié | 8, 9, 12 |
| 6. Tableaux globaux et par catégorie | 8, 12 |
| 7. Intervalles de confiance et tests statistiques | 15 |
| 8. Coûts : temps, mémoire GPU, complexité | 14 |
| 9. Matrices de confusion et ROC | 8 |
| 10. Comparaisons visuelles | 10 |
| 11. Analyse des erreurs | 11 |
| 12. Pourquoi U-Net++ a le meilleur Dice global sur BUSI à l'étape 1 | 18 |
| 13. Conclusion comparative | 20 |

**À exécuter sur le PC avec GPU**, depuis le dossier `medvision-smart`, avec le dossier `runs/` complet (60 fichiers `best.pth`).

---
## 0. Préparation

Le notebook **réutilise le code du projet**, pour garantir que l'évaluation se fait exactement comme l'entraînement :

| Réutilisé | Fichier | Rôle |
|---|---|---|
| `creer_modele`, `complexite` | `models.py` | mêmes architectures que les poids |
| `lire_paire`, `vers_tenseur` | `train.py` | même lecture, redimensionnement 256 × 256, binarisation `> 127` et normalisation ImageNet |
| `otsu_parametre`, `MEILLEUR_REGLAGE` | `otsu_baseline.py` | **la** fonction Otsu de Hiba, sans réécriture |

In [ ]:
import os, json, glob, time, random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import roc_auc_score, roc_curve
import torch

from models import MODELES, NOMS, creer_modele, complexite
from train import lire_paire, vers_tenseur, SPLITS
from otsu_baseline import otsu_parametre, MEILLEUR_REGLAGE

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATASETS = ["busi", "kvasir"]
NOM_DS = {"busi": "BUSI", "kvasir": "Kvasir-SEG"}
GRAINES = [0, 1, 2, 3, 4]
RUNS = "runs"
OUT = "outputs/v2"
os.makedirs(OUT, exist_ok=True)

ALGOS = ["otsu"] + MODELES
NOMS_ALGO = {"otsu": "Otsu", **NOMS}
CAT_COL = {"busi": "classe", "kvasir": "taille_cat"}
ORDRE_CAT = {"busi": ["benign", "malignant", "normal"], "kvasir": ["petit", "moyen", "grand"]}
COUL = {"otsu": "#9AA0A6", "cnn_simple": "#E8A33D", "unet": "#2F6DB5", "unetpp": "#7B4FB0",
        "attention_unet": "#2CA58D", "deeplabv3plus": "#D1495B", "segformer": "#1B998B"}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sns.set_theme(style="whitegrid")

def pm(s, k=3):
    # Moyenne ± écart-type (écart-type nul s'il n'y a qu'une valeur).
    s = pd.Series(s).dropna()
    return f"{s.mean():.{k}f} ± {s.std(ddof=1) if len(s) > 1 else 0:.{k}f}"

print("Calcul sur :", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")

---
## 1. Les entraînements : hyperparamètres, complexité, coûts

On relit les 60 fichiers `resume.json` écrits par `train.py`. On vérifie d'abord qu'aucun entraînement ne manque.

In [ ]:
resumes = pd.DataFrame([json.load(open(f, encoding="utf-8"))
                        for f in glob.glob(os.path.join(RUNS, "*", "*", "graine*", "resume.json"))])
manquants = [(d, m, g) for d in DATASETS for m in MODELES for g in GRAINES
             if not os.path.exists(os.path.join(RUNS, d, m, f"graine{g}", "best.pth"))]
print(f"Entraînements trouvés : {len(resumes)} / {len(DATASETS) * len(MODELES) * len(GRAINES)}")
print("Poids manquants :", manquants if manquants else "aucun ✅")

### 1.1 Protocole d'entraînement (remarque 5)

Tous les réseaux partagent le même protocole ; seul le learning rate dépend du modèle (réglé sur la validation).

In [ ]:
proto = (resumes.groupby(["modele"])
         .agg(optimiseur=("optimiseur", "first"), perte=("perte", "first"), batch=("batch", "first"),
              taille=("taille", "first"), epoques_max=("epoques_max", "first"), patience=("patience", "first"),
              encodeur_pretraine=("encodeur_pretraine", "first"))
         .reindex(MODELES))
lr = resumes.pivot_table(index="modele", columns="dataset", values="lr", aggfunc="first").reindex(MODELES)
lr.columns = [f"lr {NOM_DS[c]}" for c in lr.columns]
proto = proto.join(lr)
proto.index = [NOMS[m] for m in proto.index]
display(proto)
print("Augmentation :", resumes["augmentation"].iloc[0], "| normalisation :", resumes["normalisation"].iloc[0])
proto.to_csv(f"{OUT}/protocole_entrainement.csv")

### 1.2 Complexité et résultats de validation (5 graines)

Moyenne ± écart-type sur les 5 graines. *Couches* = couches à poids (convolutions, convolutions transposées, linéaires).

In [ ]:
lignes = []
for (d, m), g in resumes.groupby(["dataset", "modele"]):
    lignes.append({"dataset": NOM_DS[d], "modèle": NOMS[m],
                   "paramètres (M)": round(g["n_parametres"].iloc[0] / 1e6, 2), "couches": int(g["n_couches"].iloc[0]),
                   "Dice val": pm(g["dice_val"]), "IoU val": pm(g["iou_val"]),
                   "meilleure époque": pm(g["meilleure_epoque"], 1),
                   "entraînement (min)": pm(g["temps_entrainement_s"] / 60, 1),
                   "mémoire GPU (Mo)": pm(g["memoire_gpu_max_mo"], 0) if g["memoire_gpu_max_mo"].notna().any() else "—"})
recap_val = pd.DataFrame(lignes)
recap_val["_o"] = recap_val["modèle"].map({NOMS[m]: i for i, m in enumerate(MODELES)})
recap_val = recap_val.sort_values(["dataset", "_o"]).drop(columns="_o").set_index(["dataset", "modèle"])
display(recap_val)
recap_val.to_csv(f"{OUT}/recapitulatif_validation.csv")

---
## 2. Courbes d'apprentissage (remarque 6)

Pour chaque réseau : **perte d'entraînement et de validation** (à gauche) et **Dice et IoU de validation** (à droite), en moyenne sur les 5 graines, avec une bande d'un écart-type. Les entraînements s'arrêtent à des époques différentes selon la graine (arrêt anticipé) : à chaque époque, la moyenne porte sur les graines encore en cours.

In [ ]:
hist = []
for d in DATASETS:
    for m in MODELES:
        for g in GRAINES:
            f = os.path.join(RUNS, d, m, f"graine{g}", "historique.csv")
            if os.path.exists(f):
                hist.append(pd.read_csv(f).assign(dataset=d, modele=m, graine=g))
hist = pd.concat(hist, ignore_index=True)

def bande(ax, h, col, couleur, label, ls="-"):
    s = h.groupby("epoque")[col].agg(["mean", "std"]).fillna(0)
    ax.plot(s.index, s["mean"], color=couleur, ls=ls, lw=1.6, label=label)
    ax.fill_between(s.index, s["mean"] - s["std"], s["mean"] + s["std"], color=couleur, alpha=0.15)

for d in DATASETS:
    fig, axes = plt.subplots(2, len(MODELES), figsize=(4 * len(MODELES), 6.5), sharex="col")
    for k, m in enumerate(MODELES):
        h = hist[(hist.dataset == d) & (hist.modele == m)]
        bande(axes[0, k], h, "perte_train", "#555555", "perte train")
        bande(axes[0, k], h, "perte_val", "#2F6DB5", "perte val")
        bande(axes[1, k], h, "dice_val", "#2CA58D", "Dice val")
        bande(axes[1, k], h, "iou_val", "#D1495B", "IoU val", ls="--")
        axes[0, k].set_title(NOMS[m])
        axes[1, k].set_ylim(0, 1)
        axes[1, k].set_xlabel("époque")
    axes[0, 0].set_ylabel("perte (BCE + Dice)")
    axes[1, 0].set_ylabel("score de validation")
    axes[0, 0].legend(fontsize=8)
    axes[1, 0].legend(fontsize=8)
    fig.suptitle(f"Courbes d'apprentissage — {NOM_DS[d]} (moyenne ± écart-type sur 5 graines)", fontsize=13)
    plt.tight_layout()
    plt.savefig(f"{OUT}/courbes_apprentissage_{d}.png", dpi=150)
    plt.show()

**Lecture :** la perte d'entraînement baisse jusqu'au bout, tandis que la perte de validation se stabilise puis remonte légèrement : c'est le début du **surapprentissage**. L'arrêt anticipé garde les poids de la meilleure époque de validation, avant cette remontée.

---
## 3. Chargement des données et Otsu harmonisé (remarque 9)

Les images sont lues avec `lire_paire` de `train.py` : RGB 256 × 256, masque binaire. Otsu travaille sur la même image convertie en niveaux de gris (pour BUSI, les trois canaux sont identiques : c'est exactement l'image d'origine redimensionnée).

In [ ]:
def charger(dataset, split):
    df = pd.read_csv(SPLITS[dataset])
    df = df[df["split"] == split].reset_index(drop=True)
    paires = [lire_paire(l, 256) for _, l in df.iterrows()]
    X = np.stack([p[0] for p in paires])
    Y = np.stack([p[1] for p in paires]).astype(bool)
    G = np.stack([cv2.cvtColor(x, cv2.COLOR_RGB2GRAY) for x in X])
    return df, X, Y, G

VAL = {d: charger(d, "val") for d in DATASETS}
for d in DATASETS:
    print(f"{NOM_DS[d]} : {len(VAL[d][0])} images de validation")

### 3.1 Les métriques, vérifiées

Pour chaque image, à 256 × 256 :

- **Dice, IoU, précision, rappel** : définitions classiques. Une image sans lésion vaut 1 si rien n'est prédit, 0 sinon ; son rappel n'est pas défini (`NaN`).
- **F1** : moyenne harmonique de la précision et du rappel. **Pour un masque binaire, le F1 d'une image est mathématiquement égal à son Dice** (2 VP / (2 VP + FP + FN)). On le calcule quand même, et on donne aussi un **F1 « micro »**, qui regroupe tous les pixels du test : il donne plus de poids aux grandes lésions.
- **mAP, vérifié de deux façons** (remarque 9) :
  - `map_img` : la version utilisée jusqu'ici, la proportion des seuils d'IoU (0,50 ; 0,55 ; … ; 0,95) atteints par l'image ;
  - `map_inst` : la définition standard **par instances** (Data Science Bowl 2018). Chaque zone connexe est une lésion ; à chaque seuil t, une lésion prédite est un vrai positif si elle recouvre une lésion réelle avec un IoU > t ; on calcule VP / (VP + FP + FN), puis la moyenne sur les seuils.

  Les deux définitions coïncident quand il y a une seule lésion et une seule prédiction ; la seconde **pénalise en plus les zones parasites et les lésions oubliées**.

In [ ]:
SEUILS = np.linspace(0.5, 0.95, 10)

def map_instances(p, v):
    # mAP par instances (Data Science Bowl 2018) : zones connexes appariées par IoU > t.
    n_p, lp = cv2.connectedComponents(p.astype(np.uint8))
    n_v, lv = cv2.connectedComponents(v.astype(np.uint8))
    if n_p == 1 and n_v == 1:
        return 1.0
    if n_p == 1 or n_v == 1:
        return 0.0
    inter = np.zeros((n_p, n_v))
    np.add.at(inter, (lp.ravel(), lv.ravel()), 1)
    aire_p, aire_v = inter.sum(1), inter.sum(0)
    inter = inter[1:, 1:]
    iou = inter / (aire_p[1:, None] + aire_v[None, 1:] - inter)
    scores = []
    for t in SEUILS:
        m = iou > t                                   # t >= 0,5 : appariement unique
        vp = int(m.any(axis=1).sum())
        fp, fn = (n_p - 1) - vp, (n_v - 1) - int(m.any(axis=0).sum())
        scores.append(vp / (vp + fp + fn))
    return float(np.mean(scores))


def metriques_image(p, v):
    vp, fp, fn = (p & v).sum(), (p & ~v).sum(), (~p & v).sum()
    if v.sum() == 0:
        ok = float(p.sum() == 0)
        return {"dice": ok, "iou": ok, "precision": ok, "rappel": np.nan, "f1": ok,
                "map_img": ok, "map_inst": map_instances(p, v)}
    iou, dice = vp / (vp + fp + fn), 2 * vp / (2 * vp + fp + fn)
    prec, rap = (vp / (vp + fp) if vp + fp else 0.0), vp / (vp + fn)
    return {"dice": dice, "iou": iou, "precision": prec, "rappel": rap,
            "f1": 2 * prec * rap / (prec + rap) if prec + rap else 0.0,
            "map_img": float(np.mean(iou >= SEUILS)), "map_inst": map_instances(p, v)}

# Vérifications sur des cas connus
carre = np.zeros((20, 20), bool); carre[5:15, 5:15] = True
moitie = carre.copy(); moitie[5:15, 10:15] = False
deux = carre.copy(); deux[0:2, 0:2] = True
for nom, p, v in [("parfait", carre, carre), ("moitié trouvée", moitie, carre),
                  ("+ zone parasite", deux, carre), ("rien trouvé", np.zeros_like(carre), carre),
                  ("image saine, rien prédit", np.zeros_like(carre), np.zeros_like(carre))]:
    r = metriques_image(p, v)
    print(f"{nom:26s}", {k: (round(x, 3) if x == x else "NaN") for k, x in r.items()})

**Lecture des vérifications :** le Dice et le F1 sont identiques à chaque fois, ce qui confirme l'égalité. Avec une zone parasite, `map_img` reste à 1 (l'IoU global reste élevé), alors que `map_inst` descend à 0,5 (une vraie lésion trouvée, une fausse) : c'est la différence entre les deux définitions.

### 3.2 Otsu : le code de Hiba, réglé sur la validation de chaque dataset

- **BUSI** : on utilise `MEILLEUR_REGLAGE` (flou 3, nettoyage 9, zone `sans_bord`), choisi par Hiba sur la validation ; on vérifie qu'on retrouve son Dice de validation de **0,201**.
- **Kvasir-SEG** : le réglage de BUSI a été choisi pour des tumeurs **sombres**. Les polypes ne sont pas forcément plus sombres que la paroi de l'intestin. On règle donc Otsu **sur la validation de Kvasir**, avec les mêmes 27 combinaisons, plus la **polarité** : garder les zones sombres (fonction telle quelle) ou les zones claires (fonction appliquée à l'image inversée, 255 − image). Soit 54 combinaisons.

In [ ]:
def otsu(G, reglage):
    pol = reglage.get("polarite", "sombre")
    r = {k: v for k, v in reglage.items() if k != "polarite"}
    return np.stack([otsu_parametre(g if pol == "sombre" else 255 - g, **r) for g in G]).astype(bool)

def dice_moyen(P, Y):
    return float(np.mean([metriques_image(p, v)["dice"] for p, v in zip(P, Y)]))

REGLAGE_OTSU = {"busi": {**MEILLEUR_REGLAGE, "polarite": "sombre"}}
print(f"BUSI, réglage de Hiba {MEILLEUR_REGLAGE} : Dice val = {dice_moyen(otsu(VAL['busi'][3], REGLAGE_OTSU['busi']), VAL['busi'][2]):.3f} (attendu 0,201)")

grille = []
for pol in ["sombre", "clair"]:
    for flou in [3, 5, 9]:
        for nett in [0, 5, 9]:
            for zone in ["toutes", "plus_grande", "sans_bord"]:
                r = {"flou": flou, "nettoyage": nett, "zone": zone, "polarite": pol}
                grille.append({**r, "dice_val": dice_moyen(otsu(VAL["kvasir"][3], r), VAL["kvasir"][2])})
grille = pd.DataFrame(grille).sort_values("dice_val", ascending=False)
grille.to_csv(f"{OUT}/reglage_otsu_kvasir_val.csv", index=False)
display(grille.head(8).round(3))
REGLAGE_OTSU["kvasir"] = grille.iloc[0][["flou", "nettoyage", "zone", "polarite"]].to_dict()
REGLAGE_OTSU["kvasir"].update({"flou": int(REGLAGE_OTSU["kvasir"]["flou"]), "nettoyage": int(REGLAGE_OTSU["kvasir"]["nettoyage"])})
print("Réglage retenu pour Kvasir-SEG :", REGLAGE_OTSU["kvasir"])

---
## 4. Contrôle de cohérence (validation)

Avant de toucher au test : chacun des 60 réseaux rechargé doit redonner **le Dice de validation enregistré pendant son entraînement**. Un écart important signalerait un mauvais fichier de poids ou un prétraitement différent.

In [ ]:
def charger_reseau(d, m, g):
    net = creer_modele(m, pretrained=False)
    etat = torch.load(os.path.join(RUNS, d, m, f"graine{g}", "best.pth"), map_location=DEVICE)
    net.load_state_dict({k: v.float() for k, v in etat.items()})
    return net.to(DEVICE).eval()

@torch.no_grad()
def predire_probs(net, X, bs=16):
    out = []
    for k in range(0, len(X), bs):
        x, _ = vers_tenseur(X[k:k + bs], np.zeros(X[k:k + bs].shape[:3], np.uint8), DEVICE)
        out.append(torch.sigmoid(net(x))[:, 0].float().cpu().numpy())
    return np.concatenate(out)

controle = []
for d in DATASETS:
    _, Xv, Yv, _ = VAL[d]
    for m in MODELES:
        for g in GRAINES:
            net = charger_reseau(d, m, g)
            dv = dice_moyen(predire_probs(net, Xv) > 0.5, Yv)
            att = resumes.query("dataset == @d and modele == @m and graine == @g")["dice_val"].iloc[0]
            controle.append({"dataset": d, "modele": m, "graine": g, "ecart": abs(dv - att)})
            del net
controle = pd.DataFrame(controle)
synthese = controle.groupby(["dataset", "modele"])["ecart"].max().unstack(0).reindex(MODELES)
synthese.index = [NOMS[m] for m in synthese.index]
display(synthese.round(4).rename(columns=NOM_DS))
print("✅ Tous les réseaux sont cohérents (écart ≤ 0,01)" if controle["ecart"].max() <= 0.01
      else "⚠️ Écart > 0,01 : vérifier les poids concernés avant d'aller plus loin")

---
## 5. Évaluation finale sur le test

🔒 **Seule utilisation du test dans tout le projet.**

Pour chaque dataset : Otsu (une fois), puis chaque réseau avec ses 5 graines. Pour chaque image, on calcule les métriques de la partie 3.1. On garde aussi, par entraînement, les comptes de pixels (pour la matrice de confusion et le F1 micro) et l'**AUC au niveau des pixels**. Les probabilités de la graine 0 sont gardées pour les figures.

In [ ]:
TEST, PROBS0, PRED_OTSU = {}, {}, {}
lignes, runs = [], []

def comptes(B, Y):
    return {"vp": int((B & Y).sum()), "fp": int((B & ~Y).sum()), "fn": int((~B & Y).sum()), "vn": int((~B & ~Y).sum())}

for d in DATASETS:
    df, X, Y, G = charger(d, "test")
    TEST[d] = (df, X, Y, G)
    cat = df[CAT_COL[d]].values
    print(f"\n{NOM_DS[d]} : {len(df)} images de test", flush=True)

    B = otsu(G, REGLAGE_OTSU[d])
    PRED_OTSU[d] = B
    for i in range(len(df)):
        lignes.append({"dataset": d, "modele": "otsu", "graine": 0, "id": df["id"][i], "categorie": cat[i],
                       **metriques_image(B[i], Y[i])})
    runs.append({"dataset": d, "modele": "otsu", "graine": 0, "auc_pixels": np.nan, **comptes(B, Y)})

    for m in MODELES:
        for g in GRAINES:
            net = charger_reseau(d, m, g)
            P = predire_probs(net, X)
            del net
            B = P > 0.5
            for i in range(len(df)):
                lignes.append({"dataset": d, "modele": m, "graine": g, "id": df["id"][i], "categorie": cat[i],
                               **metriques_image(B[i], Y[i])})
            runs.append({"dataset": d, "modele": m, "graine": g,
                         "auc_pixels": roc_auc_score(Y.ravel(), P.ravel()), **comptes(B, Y)})
            if g == 0:
                PROBS0[(d, m)] = P.astype(np.float16)
        print(f"  {NOMS[m]} : 5 graines évaluées", flush=True)

res = pd.DataFrame(lignes)
runs = pd.DataFrame(runs)
runs["precision_micro"] = runs.vp / (runs.vp + runs.fp).replace(0, np.nan)
runs["rappel_micro"] = runs.vp / (runs.vp + runs.fn)
runs["f1_micro"] = 2 * runs.vp / (2 * runs.vp + runs.fp + runs.fn)
res.to_csv(f"{OUT}/resultats_test_par_image.csv", index=False)
runs.to_csv(f"{OUT}/resultats_test_par_entrainement.csv", index=False)
print(f"\nEnregistré : {len(res)} lignes (image × algorithme × graine)")

---
## 6. Résultats sur le test (remarques 8 et 12)

### 6.1 Tableau global : moyenne ± écart-type sur les 5 graines

Pour chaque graine, on calcule la moyenne des métriques sur les images du test ; le tableau donne la **moyenne et l'écart-type de ces 5 moyennes**. L'écart-type mesure donc la **stabilité d'un entraînement à l'autre**. Otsu, déterministe, n'a pas d'écart-type.

In [ ]:
METRIQUES = ["dice", "iou", "precision", "rappel", "f1", "map_img", "map_inst"]
par_graine = res.groupby(["dataset", "modele", "graine"])[METRIQUES].mean().reset_index()
par_graine = par_graine.merge(runs[["dataset", "modele", "graine", "f1_micro", "auc_pixels"]],
                              on=["dataset", "modele", "graine"])
lesions = res[res["categorie"] != "normal"].groupby(["dataset", "modele", "graine"])["dice"].mean().rename("dice_lesions")
par_graine = par_graine.merge(lesions.reset_index(), on=["dataset", "modele", "graine"])

COLS = ["dice", "dice_lesions", "iou", "precision", "rappel", "f1", "f1_micro", "map_img", "map_inst", "auc_pixels"]
TABLEAUX = {}
for d in DATASETS:
    t = {}
    for a in ALGOS:
        g = par_graine[(par_graine.dataset == d) & (par_graine.modele == a)]
        t[NOMS_ALGO[a]] = {c: (pm(g[c]) if g[c].notna().any() else "—") for c in COLS}
    TABLEAUX[d] = pd.DataFrame(t).T
    if d != "busi":
        TABLEAUX[d] = TABLEAUX[d].drop(columns="dice_lesions")   # Kvasir : toutes les images ont une lésion
    print(f"\n=== {NOM_DS[d]} — test ({len(TEST[d][0])} images) ===")
    display(TABLEAUX[d])
    TABLEAUX[d].to_csv(f"{OUT}/tableau_test_{d}.csv")

*Colonnes :* `dice_lesions` = Dice sur les seules images avec lésion (BUSI) ; `f1_micro` = F1 sur tous les pixels du test regroupés ; `map_img` et `map_inst` = les deux définitions du mAP (partie 3.1) ; `auc_pixels` = aire sous la courbe ROC au niveau des pixels.

### 6.2 Par catégorie de lésion

- **BUSI** : *benign*, *malignant*, *normal* (pour *normal*, Dice = part des images saines laissées vides ; le rappel n'est pas défini).
- **Kvasir-SEG** : polypes **petits, moyens, grands** (terciles de taille, notebook 01_kvasir).

In [ ]:
CAT_METRIQUES = ["dice", "iou", "precision", "rappel", "f1"]
par_cat = res.groupby(["dataset", "modele", "graine", "categorie"])[CAT_METRIQUES].mean().reset_index()
TAB_CAT = {}
for d in DATASETS:
    blocs = {}
    for c in ORDRE_CAT[d]:
        t = {}
        for a in ALGOS:
            g = par_cat[(par_cat.dataset == d) & (par_cat.modele == a) & (par_cat.categorie == c)]
            t[NOMS_ALGO[a]] = {mt: (pm(g[mt]) if g[mt].notna().any() else "—") for mt in CAT_METRIQUES}
        blocs[c] = pd.DataFrame(t).T
    TAB_CAT[d] = pd.concat(blocs, axis=1)
    print(f"\n=== {NOM_DS[d]} — par catégorie (moyenne ± écart-type sur 5 graines) ===")
    display(TAB_CAT[d])
    TAB_CAT[d].to_csv(f"{OUT}/tableau_test_categories_{d}.csv")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, d in zip(axes, DATASETS):
    m = par_cat[par_cat.dataset == d].groupby(["modele", "categorie"])["dice"].agg(["mean", "std"]).reset_index()
    largeur = 0.8 / len(ALGOS)
    for k, a in enumerate(ALGOS):
        s = m[m.modele == a].set_index("categorie").reindex(ORDRE_CAT[d])
        x = np.arange(len(ORDRE_CAT[d])) + (k - len(ALGOS) / 2 + 0.5) * largeur
        ax.bar(x, s["mean"], largeur, yerr=s["std"].fillna(0), color=COUL[a], label=NOMS_ALGO[a], capsize=2)
    ax.set_xticks(np.arange(len(ORDRE_CAT[d])), ORDRE_CAT[d])
    ax.set_ylim(0, 1.05)
    ax.set_title(f"{NOM_DS[d]} : Dice par catégorie")
axes[0].set_ylabel("Dice moyen (test)")
axes[1].legend(fontsize=8, ncol=2, loc="lower right")
plt.tight_layout()
plt.savefig(f"{OUT}/dice_par_categorie.png", dpi=150)
plt.show()

---
## 7. Intervalles de confiance et tests statistiques (remarque 15)

Pour ne pas confondre la variabilité des **images** et celle des **graines**, on travaille ici sur le **Dice de chaque image moyenné sur les 5 graines**.

1. **Intervalle de confiance à 95 %** du Dice moyen, par bootstrap (2 000 tirages des images avec remise).
2. **Test de Friedman** : les 7 algorithmes ont-ils tous la même performance ? (test non paramétrique sur des mesures appariées, les mêmes images pour tous).
3. **Tests de Wilcoxon appariés** entre le meilleur algorithme et chacun des autres, avec la **correction de Holm** pour tenir compte des comparaisons multiples. Un modèle dont la p-value corrigée dépasse 0,05 n'est **pas significativement différent** du meilleur.

In [ ]:
rng = np.random.default_rng(SEED)

def ic_bootstrap(v, n=2000):
    v = np.asarray(v)
    moy = v[rng.integers(0, len(v), size=(n, len(v)))].mean(axis=1)
    return np.percentile(moy, [2.5, 97.5])

def holm(p):
    p = np.asarray(p, float)
    ordre = np.argsort(p)
    corr = np.empty_like(p)
    cumul = 0
    for rang, i in enumerate(ordre):
        cumul = max(cumul, min(1.0, (len(p) - rang) * p[i]))
        corr[i] = cumul
    return corr

STATS, GROUPE_TETE = {}, {}
for d in DATASETS:
    dice_img = res[res.dataset == d].groupby(["id", "modele"])["dice"].mean().unstack()[ALGOS]
    moy = dice_img.mean().sort_values(ascending=False)
    meilleur = moy.index[0]
    fr = stats.friedmanchisquare(*[dice_img[a] for a in ALGOS])
    lignes = []
    autres = [a for a in moy.index if a != meilleur]
    p_bruts = []
    for a in autres:
        diff = dice_img[meilleur] - dice_img[a]
        p_bruts.append(stats.wilcoxon(dice_img[meilleur], dice_img[a]).pvalue if (diff != 0).any() else 1.0)
    p_holm = dict(zip(autres, holm(p_bruts)))
    for a in moy.index:
        bas, haut = ic_bootstrap(dice_img[a])
        lignes.append({"algorithme": NOMS_ALGO[a], "Dice moyen": round(moy[a], 3),
                       "IC 95 %": f"[{bas:.3f} ; {haut:.3f}]",
                       f"p (Holm) vs {NOMS_ALGO[meilleur]}": "—" if a == meilleur else f"{p_holm[a]:.4f}",
                       "différence significative": "—" if a == meilleur else ("oui" if p_holm[a] < 0.05 else "non")})
    STATS[d] = pd.DataFrame(lignes).set_index("algorithme")
    GROUPE_TETE[d] = [meilleur] + [a for a in autres if p_holm[a] >= 0.05]
    print(f"\n=== {NOM_DS[d]} === Friedman : χ² = {fr.statistic:.1f}, p = {fr.pvalue:.2e}")
    display(STATS[d])
    STATS[d].to_csv(f"{OUT}/tests_statistiques_{d}.csv")
    print("Groupe de tête (non significativement différents du meilleur) :", [NOMS_ALGO[a] for a in GROUPE_TETE[d]])

---
## 8. Coûts : temps, mémoire GPU, complexité (remarque 14)

- **Temps d'inférence** : temps pour segmenter **une** image (lot de taille 1), moyenne sur 50 images après un échauffement, en synchronisant le GPU.
- **Mémoire GPU d'inférence** : pic de mémoire pendant une prédiction d'une image (poids compris).
- **Temps d'entraînement et mémoire GPU d'entraînement** : relus dans les `resume.json` (moyenne ± écart-type sur 5 graines).

In [ ]:
def mesurer_inference(net, X, n=50):
    sync = torch.cuda.synchronize if DEVICE.type == "cuda" else (lambda: None)
    for i in range(5):
        predire_probs(net, X[i:i + 1])
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    sync()
    t0 = time.perf_counter()
    for i in range(n):
        predire_probs(net, X[i % len(X): i % len(X) + 1])
    sync()
    ms = (time.perf_counter() - t0) / n * 1000
    mem = torch.cuda.max_memory_allocated() / 2**20 if DEVICE.type == "cuda" else np.nan
    return ms, mem

couts = []
for d in DATASETS:
    _, X, _, G = TEST[d]
    t0 = time.perf_counter()
    otsu(G[:50], REGLAGE_OTSU[d])
    couts.append({"dataset": d, "modele": "otsu", "inference_ms": (time.perf_counter() - t0) / 50 * 1000,
                  "memoire_inference_mo": 0.0, "n_parametres": 0, "n_couches": 0})
    for m in MODELES:
        net = charger_reseau(d, m, 0)
        ms, mem = mesurer_inference(net, X)
        couts.append({"dataset": d, "modele": m, "inference_ms": ms, "memoire_inference_mo": mem, **complexite(net)})
        del net
couts = pd.DataFrame(couts)
entr = resumes.groupby(["dataset", "modele"]).agg(entr_min=("temps_entrainement_s", lambda s: pm(s / 60, 1)),
                                                  mem_entr=("memoire_gpu_max_mo", lambda s: pm(s, 0))).reset_index()
couts = couts.merge(entr, on=["dataset", "modele"], how="left")
couts.to_csv(f"{OUT}/couts.csv", index=False)

for d in DATASETS:
    c = couts[couts.dataset == d].set_index("modele").reindex(ALGOS)
    t = pd.DataFrame({"paramètres (M)": (c["n_parametres"] / 1e6).round(2), "couches": c["n_couches"],
                      "inférence (ms/image)": c["inference_ms"].round(1),
                      "mémoire GPU inférence (Mo)": c["memoire_inference_mo"].round(0),
                      "entraînement (min)": c["entr_min"].fillna("—"),
                      "mémoire GPU entraînement (Mo)": c["mem_entr"].fillna("—")})
    t.index = [NOMS_ALGO[a] for a in t.index]
    print(f"\n=== {NOM_DS[d]} ===")
    display(t)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, d in zip(axes, DATASETS):
    c = couts[couts.dataset == d].set_index("modele")
    dm = par_graine[par_graine.dataset == d].groupby("modele")["dice"].agg(["mean", "std"])
    for a in ALGOS:
        ax.errorbar(c.loc[a, "inference_ms"], dm.loc[a, "mean"], yerr=dm.loc[a, "std"], fmt="o",
                    ms=6 + c.loc[a, "n_parametres"] / 2e6, color=COUL[a], capsize=3)
        ax.annotate(NOMS_ALGO[a], (c.loc[a, "inference_ms"], dm.loc[a, "mean"]), xytext=(6, -4),
                    textcoords="offset points", fontsize=9)
    ax.set_xscale("log")
    ax.set_xlabel("temps d'inférence par image (ms, échelle log)")
    ax.set_ylabel("Dice test (moyenne ± écart-type, 5 graines)")
    ax.set_title(f"{NOM_DS[d]} : qualité, vitesse et taille (taille du point = paramètres)")
plt.tight_layout()
plt.savefig(f"{OUT}/compromis_qualite_vitesse.png", dpi=150)
plt.show()

---
## 9. Matrices de confusion et courbes ROC

**Matrices de confusion au niveau des pixels**, normalisées par ligne et **cumulées sur les 5 graines** : la case « lésion / prédit lésion » est le rappel pixel, la case « fond / prédit fond » la spécificité.

In [ ]:
fig, axes = plt.subplots(len(DATASETS), len(ALGOS), figsize=(2.6 * len(ALGOS), 2.8 * len(DATASETS)))
for i, d in enumerate(DATASETS):
    for k, a in enumerate(ALGOS):
        s = runs[(runs.dataset == d) & (runs.modele == a)][["vn", "fp", "fn", "vp"]].sum()
        mc = np.array([[s.vn, s.fp], [s.fn, s.vp]], float)
        mc = mc / mc.sum(axis=1, keepdims=True)
        sns.heatmap(mc, annot=True, fmt=".1%", cmap="Blues", vmin=0, vmax=1, cbar=False, ax=axes[i, k],
                    xticklabels=["fond", "lésion"], yticklabels=["fond", "lésion"], annot_kws={"size": 9})
        axes[i, k].set_title(f"{NOM_DS[d]} · {NOMS_ALGO[a]}", fontsize=9)
        axes[i, k].set_xlabel("prédit" if i == len(DATASETS) - 1 else "")
        axes[i, k].set_ylabel("réel" if k == 0 else "")
plt.tight_layout()
plt.savefig(f"{OUT}/matrices_confusion_pixels.png", dpi=150)
plt.show()

**Courbes ROC au niveau des pixels** (graine 0 ; l'AUC du tableau 6.1 est la moyenne ± écart-type sur les 5 graines). Otsu, qui ne donne que 0 ou 1, est un point.

Pour **BUSI**, qui contient des images saines, on ajoute la **matrice de confusion au niveau des images** : la lésion est-elle détectée (au moins un pixel prédit) ? Elle n'a pas de sens pour Kvasir-SEG, où toutes les images contiennent un polype.

In [ ]:
fig, axes = plt.subplots(1, len(DATASETS), figsize=(13, 5.5))
for ax, d in zip(axes, DATASETS):
    Y = TEST[d][2].ravel()
    for m in MODELES:
        fpr, tpr, _ = roc_curve(Y, PROBS0[(d, m)].astype(np.float32).ravel())
        a = runs[(runs.dataset == d) & (runs.modele == m)]["auc_pixels"]
        ax.plot(fpr, tpr, color=COUL[m], lw=1.6, label=f"{NOMS[m]} (AUC {pm(a)})")
    s = runs[(runs.dataset == d) & (runs.modele == "otsu")].iloc[0]
    ax.scatter(s.fp / (s.fp + s.vn), s.vp / (s.vp + s.fn), color=COUL["otsu"], edgecolor="black", s=60, zorder=3, label="Otsu")
    ax.plot([0, 1], [0, 1], "--", color="grey", lw=1)
    ax.set_xlabel("taux de faux positifs (pixels)")
    ax.set_ylabel("taux de vrais positifs (pixels)")
    ax.set_title(f"ROC pixels — {NOM_DS[d]}")
    ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(f"{OUT}/courbes_roc_pixels.png", dpi=150)
plt.show()

# BUSI : détection au niveau des images (graine 0)
d = "busi"
y_img = TEST[d][2].reshape(len(TEST[d][2]), -1).any(axis=1)
fig, axes = plt.subplots(1, len(ALGOS), figsize=(2.6 * len(ALGOS), 2.8))
for ax, a in zip(axes, ALGOS):
    pred = (PRED_OTSU[d] if a == "otsu" else PROBS0[(d, a)].astype(np.float32) > 0.5).reshape(len(y_img), -1).any(axis=1)
    mc = np.array([[(~y_img & ~pred).sum(), (~y_img & pred).sum()], [(y_img & ~pred).sum(), (y_img & pred).sum()]])
    sns.heatmap(mc, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["rien", "lésion"], yticklabels=["saine", "lésion"])
    ax.set_title(NOMS_ALGO[a], fontsize=9)
fig.suptitle("BUSI : détection d'une lésion au niveau des images (test, graine 0)")
plt.tight_layout()
plt.savefig(f"{OUT}/matrices_confusion_images_busi.png", dpi=150)
plt.show()

---
## 10. Comparaisons visuelles (remarque 10)

Deux images par catégorie, tirées au hasard dans le test. Colonnes : l'image, le masque réel, puis les prédictions d'Otsu et des 6 réseaux (graine 0). Contour **vert** = vérité, zone **rouge** = prédiction ; le Dice est indiqué au-dessus.

In [ ]:
def montrer(ax, img, verite, pred=None, titre=""):
    ax.imshow(img)
    ax.contour(verite, levels=[0.5], colors="lime", linewidths=1.2)
    if pred is not None and pred.any():
        ax.imshow(np.ma.masked_where(~pred, pred.astype(float)), cmap="autumn", alpha=0.45, vmin=0, vmax=1)
    ax.set_title(titre, fontsize=8)
    ax.axis("off")

for d in DATASETS:
    df, X, Y, _ = TEST[d]
    idx = np.concatenate([df.index[df[CAT_COL[d]] == c].to_series().sample(2, random_state=SEED).values
                          for c in ORDRE_CAT[d]])
    fig, axes = plt.subplots(len(idx), 2 + len(ALGOS), figsize=(2.1 * (2 + len(ALGOS)), 2.2 * len(idx)))
    for r, i in enumerate(idx):
        montrer(axes[r, 0], X[i], Y[i], titre=f"{df[CAT_COL[d]][i]}")
        axes[r, 1].imshow(Y[i], cmap="gray"); axes[r, 1].axis("off"); axes[r, 1].set_title("masque réel", fontsize=8)
        for k, a in enumerate(ALGOS):
            p = PRED_OTSU[d][i] if a == "otsu" else PROBS0[(d, a)][i].astype(np.float32) > 0.5
            montrer(axes[r, 2 + k], X[i], Y[i], p, f"{NOMS_ALGO[a]} · {metriques_image(p, Y[i])['dice']:.2f}")
    fig.suptitle(f"{NOM_DS[d]} : image, masque réel et prédictions (vert = vérité, rouge = prédiction)", fontsize=12)
    plt.tight_layout()
    plt.savefig(f"{OUT}/comparaison_visuelle_{d}.png", dpi=120)
    plt.show()

---
## 11. Analyse des erreurs (remarque 11)

Trois types d'erreurs, moyennés sur les 5 graines :

- **Lésions non détectées** : images avec lésion où le Dice vaut 0 (rien de prédit, ou prédit à côté).
- **Faux positifs** : part des pixels de fond coloriés à tort et, pour BUSI, images **saines** où une lésion est signalée.
- **Mauvaises segmentations** : lésion trouvée mais contour très imparfait (0 < Dice < 0,5), et tendance à **sous-segmenter** (prédiction plus petite que la vérité) ou **sur-segmenter**, mesurée par le rapport médian surface prédite / surface réelle.

In [ ]:
erreurs = []
for d in DATASETS:
    r = res[res.dataset == d]
    les = r[r.categorie != "normal"]
    for a in ALGOS:
        for g in (GRAINES if a != "otsu" else [0]):
            x = les[(les.modele == a) & (les.graine == g)]
            ligne = {"dataset": d, "modele": a, "graine": g,
                     "lesions_manquees": int((x.dice == 0).sum()), "n_lesions": len(x),
                     "mauvaises_segmentations": int(((x.dice > 0) & (x.dice < 0.5)).sum())}
            if d == "busi":
                n = r[(r.modele == a) & (r.graine == g) & (r.categorie == "normal")]
                ligne["fausses_alertes_saines"] = int((n.dice == 0).sum())
                ligne["n_saines"] = len(n)
            s = runs[(runs.dataset == d) & (runs.modele == a) & (runs.graine == g)].iloc[0]
            ligne["fp_pixels_%"] = 100 * s.fp / (s.fp + s.vn)
            ligne["rapport_surface"] = (s.vp + s.fp) / (s.vp + s.fn)
            erreurs.append(ligne)
erreurs = pd.DataFrame(erreurs)
erreurs.to_csv(f"{OUT}/analyse_erreurs.csv", index=False)

for d in DATASETS:
    e = erreurs[erreurs.dataset == d]
    t = {}
    for a in ALGOS:
        x = e[e.modele == a]
        t[NOMS_ALGO[a]] = {"lésions non détectées": f"{pm(x.lesions_manquees, 1)} / {x.n_lesions.iloc[0]}",
                           "mauvaises segmentations (Dice < 0,5)": pm(x.mauvaises_segmentations, 1),
                           "fond colorié à tort (%)": pm(x["fp_pixels_%"], 2),
                           "surface prédite / réelle": pm(x.rapport_surface, 2)}
        if d == "busi":
            t[NOMS_ALGO[a]]["fausses alertes (images saines)"] = f"{pm(x.fausses_alertes_saines, 1)} / {x.n_saines.iloc[0]}"
    print(f"\n=== {NOM_DS[d]} ===")
    display(pd.DataFrame(t).T)

**Les cas les plus difficiles :** les 6 images avec lésion où le Dice moyen **de tous les réseaux** (5 graines) est le plus bas. Ce sont des difficultés liées à l'image elle-même, pas à un modèle. Contour vert = vérité ; en rouge, la prédiction du réseau le mieux classé (graine 0).

In [ ]:
for d in DATASETS:
    df, X, Y, _ = TEST[d]
    r = res[(res.dataset == d) & (res.modele != "otsu") & (res.categorie != "normal")]
    dur = r.groupby("id")["dice"].mean().nsmallest(6)
    meilleur = GROUPE_TETE[d][0] if GROUPE_TETE[d][0] != "otsu" else MODELES[1]
    pos = {i: k for k, i in enumerate(df["id"])}
    fig, axes = plt.subplots(1, 6, figsize=(18, 3.4))
    for ax, (i, dm) in zip(axes, dur.items()):
        k = pos[i]
        p = PROBS0[(d, meilleur)][k].astype(np.float32) > 0.5
        montrer(ax, X[k], Y[k], p, f"{df[CAT_COL[d]][k]} · lésion {Y[k].mean() * 100:.1f} %\nDice moyen des réseaux {dm:.2f}")
    fig.suptitle(f"{NOM_DS[d]} : les cas les plus difficiles (prédiction de {NOMS[meilleur]})", fontsize=12)
    plt.tight_layout()
    plt.savefig(f"{OUT}/cas_difficiles_{d}.png", dpi=120)
    plt.show()

*À commenter après exécution, en regardant les figures :* lésion qui touche le bord de l'image, contraste faible, zone voisine confondue avec la lésion, très petite lésion, annotation discutable, reflets lumineux (Kvasir-SEG)…

---
## 12. Pourquoi U-Net++ avait le meilleur Dice global sur BUSI à l'étape 1 (remarque 18)

Sur BUSI, le **Dice global** mélange deux types d'images :

- les **images avec lésion** (93), où le Dice mesure la qualité du contour ;
- les **images saines** (18), où le Dice vaut **1 ou 0** selon que le modèle reste muet ou signale une fausse lésion.

Le Dice global vaut donc (93 × Dice sur les lésions + 18 × part des images saines laissées vides) / 111. Une seule fausse alerte sur une image saine coûte 1/111 ≈ **0,009** de Dice global, autant que de dégrader de 0,01 le contour de toutes les lésions. Un modèle plus « prudent » sur les images saines peut donc gagner en Dice global tout en étant moins bon sur les lésions. La cellule suivante fait cette décomposition pour chaque modèle.

In [ ]:
r = res[res.dataset == "busi"]
deco = []
for a in ALGOS:
    x = r[r.modele == a]
    g = x.groupby("graine")
    deco.append({"algorithme": NOMS_ALGO[a],
                 "Dice global": pm(g["dice"].mean()),
                 "Dice sur les lésions": pm(x[x.categorie != "normal"].groupby("graine")["dice"].mean()),
                 "images saines laissées vides": pm(x[x.categorie == "normal"].groupby("graine")["dice"].mean()),
                 "fausses alertes (sur 18)": pm(x[x.categorie == "normal"].groupby("graine")["dice"].apply(lambda s: (s == 0).sum()), 1)})
deco = pd.DataFrame(deco).set_index("algorithme")
display(deco)
deco.to_csv(f"{OUT}/decomposition_dice_busi.csv")

**Lecture :** on compare, pour chaque modèle, la colonne « Dice sur les lésions » (qualité de segmentation) et la colonne « images saines laissées vides » (prudence). À l'étape 1, U-Net++ faisait 3 fausses alertes sur les images saines contre 6 pour U-Net, soit environ +0,027 de Dice global, alors qu'U-Net était meilleur sur les lésions (0,741 contre 0,736). **Le Dice global n'est donc pas le bon critère seul pour une aide au diagnostic** : il faut regarder séparément la segmentation des lésions et les fausses alertes.

---
## 13. Conclusion comparative (remarque 20)

Le tableau final classe les algorithmes selon les trois critères demandés : **précision de segmentation** (Dice test, 5 graines), **complexité** (paramètres) et **temps d'inférence**. Le **meilleur compromis** est le modèle le plus rapide parmi ceux qui ne sont **pas significativement moins bons** que le meilleur (groupe de tête, partie 7).

In [ ]:
for d in DATASETS:
    dm = par_graine[par_graine.dataset == d].groupby("modele")["dice"].agg(["mean", "std"])
    c = couts[couts.dataset == d].set_index("modele")
    t = pd.DataFrame({"Dice test": [pm(par_graine[(par_graine.dataset == d) & (par_graine.modele == a)]["dice"]) for a in ALGOS],
                      "rang Dice": dm["mean"].rank(ascending=False).reindex(ALGOS).astype(int).values,
                      "paramètres (M)": (c["n_parametres"] / 1e6).round(2).reindex(ALGOS).values,
                      "inférence (ms)": c["inference_ms"].round(1).reindex(ALGOS).values,
                      "groupe de tête": ["oui" if a in GROUPE_TETE[d] else "" for a in ALGOS]},
                     index=[NOMS_ALGO[a] for a in ALGOS]).sort_values("rang Dice")
    print(f"\n=== {NOM_DS[d]} ===")
    display(t)
    t.to_csv(f"{OUT}/conclusion_{d}.csv")
    meilleur = dm["mean"].idxmax()
    compromis = min(GROUPE_TETE[d], key=lambda a: c.loc[a, "inference_ms"])
    leger = min(GROUPE_TETE[d], key=lambda a: c.loc[a, "n_parametres"])
    print(f"Meilleure précision : {NOMS_ALGO[meilleur]} (Dice {dm.loc[meilleur, 'mean']:.3f} ± {(0 if pd.isna(dm.loc[meilleur, 'std']) else dm.loc[meilleur, 'std']):.3f})")
    print(f"Groupe de tête      : {', '.join(NOMS_ALGO[a] for a in GROUPE_TETE[d])}")
    print(f"Le plus rapide du groupe de tête : {NOMS_ALGO[compromis]} ({c.loc[compromis, 'inference_ms']:.1f} ms/image)")
    print(f"Le plus léger du groupe de tête  : {NOMS_ALGO[leger]} ({c.loc[leger, 'n_parametres'] / 1e6:.1f} M paramètres)")

### Synthèse (à rédiger avec les chiffres ci-dessus)

1. **Le deep learning est indispensable** sur les deux datasets : Otsu reste très loin derrière.
2. **Les encodeurs pré-entraînés font la différence** : le CNN simple, entraîné de zéro, est nettement en dessous des 5 autres réseaux.
3. **BUSI** : les modèles pré-entraînés forment un groupe de tête dont les écarts sont du même ordre que la variation d'une graine à l'autre → choisir selon le coût et le comportement sur les lésions.
4. **Kvasir-SEG** : vérifier si SegFormer reste significativement devant sur le test, comme sur la validation.
5. **Modèle recommandé** : celui du meilleur compromis précision / vitesse / complexité, en tenant compte du rappel sur les lésions (critère médical prioritaire).